# SpaceJEPA v4 · Why is v2 better?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/space-jepa4/StochasticEntrySim/ReEntryAI_JEPA_v4.ipynb)

**How to run.** Open in Colab (badge above), choose an **L4 GPU** runtime (it opens that way), and
**Runtime → Run all**. That is all. Colab can disconnect when idle or when its session limit ends;
**after any disconnect, Run all again**: every stage reads Drive, skips finished work in seconds, and
resumes what is half done. No cell ever needs editing to resume. The notebook **disconnects itself**
when the last cell finishes or a cell fails (so it never idles on paid compute); set
`AUTO_DISCONNECT = False` in the Settings cell to keep the runtime. For the analysis and report only,
you can switch to a cheap **CPU runtime**: the GPU stages then print what remains and skip.

v2 beats the classical predictor-corrector by 54 points of success within 5 km (v3), but it differs
from it in two things at once (the MPPI planner and the learned model), and from v1 in about eight
design changes. This study attributes the gains (`jepa/v4_plan.md`):

| | Question | Experiment |
|---|---|---|
| RQ1 | How much of v2's advantage is the planner, how much the model? Is v2's model close to a perfect one? | E1: MPPI through the classical controller's own physics (nominal and true parameters) and through v2's physics prior; the classical controller knowing the true density and mass (not L/D: its predictor resets the L/D scale, see v4_plan.md §14) |
| RQ2 | Does v2 still beat classical when it pulls no more load? | E2: v2 at 4.0 g, 3.5 g and at classical's own peak g per condition |
| RQ3 | Which v1 → v2 changes are necessary, which sufficient? | E3: 13 single-change variants of a one-member v2 (R0) and of v1, 5 seeds each |
| RQ4 | Why does v2 fail case 10? | E4: 40 diagnostic flights with the planner's decision log (descriptive) |

Everything is paired with v3's 5,120 flights: same 64 cases, 4 atmospheres, seeds, MPPI settings and
noise streams. Inputs (read-only): `MyDrive/ReEntryAI_jepa_v3/` and
`MyDrive/ReEntryAI_jepa_data/wide_6000/`. Output: **`MyDrive/ReEntryAI_jepa_v4/`** (layout in
`v4_plan.md` §3 and the auto-written `00_study/README.md`). v4 refuses to write anywhere else.

**Rough cost on an L4** (from v3's measured rates; the status cell measures the real ones): E0 ≈ 1 h,
E1 ≈ 4–7 h (the classical oracle is ≈ 6 CPU-hours), E2 ≈ 3–5 h, E4 ≈ 5 min, E3 training ≈ 17–20 h (70
runs), E3 flights ≈ 15–25 h, open loop ≈ 1–2 h, analysis ≈ 15 min. Flights are limited by CPU cores
(the simulator), not the GPU. It spans many sessions; that is what the resume rule is for.

## 0. Settings

`MODE`, the stages to run this session (`RUN`), auto-disconnect, and worker counts (automatic unless you
set them). Nothing else needs editing, ever: resuming after a disconnect is Run all.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
MODE = os.environ.get("JEPA_V4_MODE", "full")     # "full": the study | "smoke": every stage, tiny, numbers meaningless
RUN = {"E0": True, "E1": True, "E2": True, "E4": True, "E3_train": True, "E3_fly": True,   # stages to run this session
       "open_loop": True, "analysis": True, "report": True}
AUTO_DISCONNECT = IN_COLAB      # release the runtime when the notebook finishes or a cell fails; False keeps it
CPU_WORKERS = None              # classical flights, one simulator per core (None = every core)
GPU_WORKERS = 8              # learned and physics flights sharing the GPU (None = 6 GB each, 2 GB spare, at most 4)
                                # (on a CPU, i.e. smoke mode: half the cores, at most 4, as v3's smoke run)

## 1. Setup

Clones (or pulls) the `space-jepa4` branch, installs `pymsis` and `reportlab`, mounts Drive, checks that
the v3 folder exists and was run with the settings v4 depends on (frozen seeds, test set, MPPI, budgets),
and freezes `00_study/study_config.json` (a later session with different settings raises instead of mixing
results). Registers the auto-disconnect hook (CLAUDE.md). Reads `00_study/` of v3; writes `00_study/` of v4.
Takes about a minute.

In [3]:
BRANCH = "space-jepa4"
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis reportlab
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    V4_ROOT = f"{DRIVE}/ReEntryAI_jepa_v4" + ("" if MODE == "full" else f"_{MODE}")
    V3_ROOT = f"{DRIVE}/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
else:
    V4_ROOT = os.environ.get("JEPA_V4_ROOT", f"jepa_runs/v4_{MODE}")
    V3_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = "jepa_runs"

import json, time
from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline
from jepa import study_v4 as S4, fly_v4 as F4, ablate_v4 as AB, analyze_v4 as A4, report_v4 as R4, train_v2 as T2

CFG = S4.StudyConfigV4.smoke() if MODE == "smoke" else S4.StudyConfigV4()
L3 = S4.v3_layout(V3_ROOT)                                            # read-only v3 folder (raises if missing)
DATA_DIRS = [c if os.path.isdir(c) else os.path.dirname(c) for c in DATA_CANDIDATES if c and os.path.exists(c)]
L = S4.LayoutV4(V4_ROOT, read_only=[V3_ROOT] + DATA_DIRS).make()      # every v4 write is checked against these
FROZEN = S4.freeze_config_v4(CFG, L, L3)                               # raises if v3's or v4's settings differ
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CPUS = os.cpu_count() or 2
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
CPU_WORKERS = CPU_WORKERS or CPUS
GPU_WORKERS = GPU_WORKERS or (max(1, min(4, CPUS, int((GPU_MEM_GB - 2) // 6))) if DEVICE == "cuda"
                              else max(1, min(4, CPUS // 2)))          # CPU (smoke): as v3's smoke run
CAN_GPU = DEVICE == "cuda" or MODE == "smoke"      # GPU stages; in smoke mode the CPU stands in for the GPU
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # inherited by the GPU workers
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
GRAD_CKPT = DEVICE == "cuda" and GPU_MEM_GB < 20   # T4: recompute activations to fit T = 96 (as v3)
torch.set_num_threads(1 if DEVICE == "cuda" else max(1, CPUS // 2))
DATA = None                                        # the 6,000 flights, loaded only if a stage needs them

def need_data():
    # Load the 6,000 flights once (copied to local disk first, as v3 does).
    global DATA, DATA_PATH
    if DATA is None:
        t0 = time.time()
        DATA_PATH = T2.local_copy(T2.find_wide_dataset(DATA_CANDIDATES, download_dir=LOCAL_DATA), LOCAL_DATA)
        DATA = AB.load_data_v4(CFG, DATA_PATH, DEVICE)
        print(f"{DATA['n_episodes']} flights: train/val/test = {len(DATA['train'])}/{len(DATA['val'])}/"
              f"{len(DATA['test'])} episodes ({time.time() - t0:.0f}s)")
    return DATA

print(f"MODE={MODE}  v4 folder={V4_ROOT}  v3 folder={V3_ROOT}  device={DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)}, {GPU_MEM_GB:.0f} GB)" if DEVICE == "cuda" else "")
      + f"  cpus={CPUS}  workers: {CPU_WORKERS} CPU, {GPU_WORKERS} GPU  git={FROZEN['git_commit'][:10]}")
if MODE == "full" and DEVICE != "cuda":
    print("No GPU: the flight, training and open-loop stages will only report what remains. "
          "Runtime -> Change runtime type -> L4 for those.")

# ---- Auto-disconnect: stop paying for the runtime once the work is done or a cell fails ----
AUTO_DISCONNECT = AUTO_DISCONNECT   # set in the Settings cell (section 0)

def release_runtime(reason):
    # Flush Drive, then release the Colab runtime so it stops using compute units.
    if not (IN_COLAB and AUTO_DISCONNECT):
        print(f"[auto-disconnect off] {reason}")
        return
    print(f"[auto-disconnect] {reason}. Flushing Drive and releasing the runtime; outputs stay visible.", flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("  Drive flush:", e)
    from google.colab import runtime
    runtime.unassign()

def _release_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is not None and not isinstance(err, KeyboardInterrupt):     # pressing stop is not a failure
        release_runtime(f"a cell failed ({type(err).__name__}: {str(err)[:200]})")

_ip = get_ipython()
for _cb in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_release_on_error":
        _ip.events.unregister("post_run_cell", _cb)
_ip.events.register("post_run_cell", _release_on_error)
print("auto-disconnect on: the runtime is released when the last cell finishes or a cell fails" if AUTO_DISCONNECT
      else "auto-disconnect off (not on Colab, or switched off): the runtime stays up until you disconnect it")

Mounted at /content/drive
MODE=full  v4 folder=/content/drive/MyDrive/ReEntryAI_jepa_v4  v3 folder=/content/drive/MyDrive/ReEntryAI_jepa_v3  device=cuda (NVIDIA L4, 24 GB)  cpus=12  workers: 12 CPU, 3 GPU  git=25658086fa
auto-disconnect on: the runtime is released when the last cell finishes or a cell fails


## 2. Status

Every stage and arm: flights done / total, the measured rate, hours left, and each model's training state.
Counts result files directory by directory (fast, no tree walk). Writes `00_study/status.json`. Seconds.

In [4]:
JOBS = S4.build_jobs(CFG, L, L3)
ST = S4.status(CFG, L, JOBS)
display(pd.DataFrame(ST["stages"]))
arms = pd.DataFrame(ST["arms"])
display(arms[arms.done < arms.total] if (arms.done < arms.total).any() else arms.groupby("stage")[["done", "total"]].sum())
tr = pd.DataFrame(ST["training"])
print(f"{ST['flights_done']}/{ST['flights_total']} flights, {ST['models_done']}/{ST['models_total']} models trained"
      + (f", ~{ST['training_hours_left']} h of training left" if ST["training_hours_left"] else "")
      + ("  (E0b failed: v3's learned arms are re-flown, see 01_checks/)" if ST["e0b_failed"] else ""))

,stage,flights_done,flights_total,flights_per_h,hours_left
0,E0,544,544,3873.4,0.0
1,E1,5376,5376,312.1,0.0
2,E2,3840,3840,54.0,0.0
3,E4,40,40,44.8,0.0
4,E3,0,17920,NaN,NaN


,stage,arm,load,done,total,flights_per_h
15,E3,R0,4.5g,0,1280,NaN
16,E3,A1_inputs_v1,4.5g,0,1280,NaN
17,E3,A1b_no_measured_aero,4.5g,0,1280,NaN
18,E3,A2_memory_35s,4.5g,0,1280,NaN
19,E3,A3_blind_prober,4.5g,0,1280,NaN
20,E3,A4_train_T24,4.5g,0,1280,NaN
21,E3,A5_sigreg_pooled,4.5g,0,1280,NaN
22,E3,A6_latent_32,4.5g,0,1280,NaN
23,E3,A7a_size_250k,4.5g,0,1280,NaN
24,E3,A7b_size_33k,4.5g,0,1280,NaN


9800/27720 flights, 16/70 models trained, ~39.86 h of training left


## 3. E0: reproducibility checks (gate everything else)

* **E0a**: v3's classical flights re-flown on 32 conditions (CPU pool): must be identical (outcome, miss
  within 1e-6 km, fuel), or the notebook **stops**. Also compares the classical oracle with classical in
  the nominal atmosphere once E1 has flown it.
* **E0b**: v3's `v2single` and `v1` (seed 0, 4.5 g, 256 conditions each) re-flown by the v4 runner from
  v3's checkpoints: outcome agreement ≥ 98% and the paired log-miss difference's 95% CI must include 0.
  Its spread is the GPU-nondeterminism floor of every v4-v3 comparison. If it fails, v3's learned arms
  are re-flown with the v4 runner (3,840 flights) and used instead; record that in `v4_plan.md` §15.
* **E0c**: the v4 runner equals v3's bit for bit on the CPU (3 flights, v1 and v2 member 0).
* **Physics fidelity**: the torch port of the classical model against `math_3d` (1,000 states), and its
  open-loop landing error on 64 recorded v3 flights (300, 200, 100 s to go).

Reads v3's flights and checkpoints; writes `01_checks/` and `04_flights/` of v4. About 1 h (E0b ≈ 30–50
min). Finished checks and flights are skipped.

In [5]:
if RUN["E0"]:
    e0_jobs = [j for j in JOBS if j["stage"] == "E0"]
    S4.run_classical_v4(e0_jobs, L, workers=CPU_WORKERS)                    # E0a: 32 classical re-flights
    E0C_OK = S4.e0c_run(CFG, L, L3)                                         # E0c: runner equivalence, CPU
    if CAN_GPU:
        S4.run_flights_v4(CFG, e0_jobs, L, device=DEVICE, workers=GPU_WORKERS)   # E0b: v3 flights re-flown
    else:
        print("E0b waits for a GPU runtime")
    PHYS = S4.physics_checks(CFG, L, L3)                                     # section 4.1.3 checks 1 and 2
    E0 = S4.e0_checks(CFG, L, L3)
    if E0["E0b"] is False:                                                   # contingency: re-fly v3's learned arms
        JOBS = S4.build_jobs(CFG, L, L3)
        if CAN_GPU:
            S4.run_flights_v4(CFG, [j for j in JOBS if j["stage"] == "E0"], L, device=DEVICE, workers=GPU_WORKERS)
    for f in ("E0a.csv", "E0c.csv"):
        if (L.checks / f).exists():
            display(pd.read_csv(L.checks / f).head(12))
    if (L.checks / "E0b_summary.json").exists():
        display(pd.DataFrame(json.loads((L.checks / "E0b_summary.json").read_text())["pairs"]))
    print("physics:", json.dumps(PHYS, indent=1)[:1500])
    print("E0:", E0)
    if E0["E0a"] is False:
        raise RuntimeError("E0a FAILED: the classical controller no longer reproduces v3's flights (01_checks/E0a.csv). "
                           "Stop and report: every v4-v3 comparison assumes the same simulator and guidance.")
    if E0C_OK is False:
        raise RuntimeError("E0c FAILED: run_lockstep_v4 differs from run_lockstep_v3 on the CPU (01_checks/E0c.csv).")

[classical] 32/32 done, 0 to fly on 12 worker(s)
[flights] 512/512 done in 2 model runs (0 runs wait for training); 3 worker(s) on cuda


,check,case,atmosphere,outcome_v3,outcome_v4,miss_v3_km,miss_v4_km,d_miss_km,fuel_v3_kg,fuel_v4_kg,d_fuel_kg,identical
0,E0a,0,nominal,drogue,drogue,2.085779,2.085779,0.0,93.631475,93.631475,0.0,True
1,E0a,0,mismatch,drogue,drogue,3.475229,3.475229,0.0,69.691809,69.691809,0.0,True
2,E0a,0,mismatch_high,drogue,drogue,1.132600,1.132600,0.0,111.935335,111.935335,0.0,True
3,E0a,0,out_of_range,drogue,drogue,32.042780,32.042780,0.0,28.619489,28.619489,0.0,True
4,E0a,8,nominal,drogue,drogue,1.954191,1.954191,0.0,95.498399,95.498399,0.0,True
5,E0a,8,mismatch,drogue,drogue,3.278419,3.278419,0.0,96.324250,96.324250,0.0,True
6,E0a,8,mismatch_high,drogue,drogue,0.859298,0.859298,0.0,111.514902,111.514902,0.0,True
7,E0a,8,out_of_range,drogue,drogue,25.150871,25.150871,0.0,26.151946,26.151946,0.0,True
8,E0a,16,nominal,drogue,drogue,6.043972,6.043972,0.0,47.794246,47.794246,0.0,True
9,E0a,16,mismatch,drogue,drogue,10.742661,10.742661,0.0,47.408849,47.408849,0.0,True


,adapter,case,atmosphere,miss_v3_km,miss_v4_km,decisions,identical,differs_in
0,v1,0,nominal,0.839596,0.839596,113,True,NaN
1,v1,16,nominal,3.154519,3.154519,109,True,NaN
2,v1,40,nominal,2.598982,2.598982,112,True,NaN
3,v2 member 0,0,nominal,2.083482,2.083482,112,True,NaN
4,v2 member 0,16,nominal,1.130592,1.130592,106,True,NaN
5,v2 member 0,40,nominal,0.966566,0.966566,114,True,NaN


,v3_arm,v4_arm,n,outcome_agreement,mean_d_log_miss,ci_lo,ci_hi,sd_d_log_miss,p05,p95,max_abs,frac_identical_miss
0,v1,v1_v4run,256,1.0,0.005553,-0.0324,0.042845,0.303552,-0.18138,0.225386,2.136567,0.019531
1,v2single,v2single_v4run,256,1.0,0.000000,0.0000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000


physics: {
 "port": {
  "n": 1000,
  "dt_s": 2.0,
  "max_rel_err": 1.8386796156240337e-15,
  "max_rel_err_increment": 2.7223540147942144e-14,
  "max_rel_err_gload": 4.782379353288882e-16,
  "max_rel_err_r": 0.0,
  "max_rel_err_phi": 0.0,
  "max_rel_err_lam": 0.0,
  "max_rel_err_V": 1.8386796156240337e-15,
  "max_rel_err_gamma": 2.2624670990420244e-16,
  "max_rel_err_chi": 2.5641596146307746e-16
 },
 "fidelity": {
  "err_truth_km": {
   "100.0": 0.292,
   "200.0": 5.46,
   "300.0": 7.795
  },
  "err_nominal_km": {
   "100.0": 0.401,
   "200.0": 6.944,
   "300.0": 8.975
  }
 }
}
E0: {'E0a': True, 'oracle_equals_classical_nominal': True, 'oracle_vs_classical_nominal_max_d_miss_km': 0.0, 'E0b': True, 'E0c': True}


## 4. E1, E2, E4: flights that need no training

* **E1** planner × model: `classical_oracle` (CPU pool, 256), and MPPI through the classical controller's
  own physics model with nominal (`mppi_nominal`, at 4.5 g and at the matched load) and true parameters
  (`mppi_truth`), and through v2's physics prior with no learning (`mppi_prior`): 5,120 flights.
* **E2** load: v2 (v3's checkpoints) at 4.0 g, 3.5 g, and the matched load (classical's measured peak g on
  each condition): 3,840 flights.
* **E4** case 10: v2 and v2single, 4.5 g and 8 g, nominal and dense air, 5 seeds, with the decision log: 40.

Run order E1 → E2 → E4, seed-major. Each flight's JSON and telemetry (`.npz`, with the decision log) is
written to `04_flights/` the moment it lands; finished flights are skipped and flights cut by a disconnect
are flown again. About 7–12 h on an L4.

In [6]:
todo = [j for j in JOBS if j["stage"] in ("E1", "E2", "E4") and RUN[j["stage"]]]
S4.run_classical_v4(todo, L, workers=CPU_WORKERS)                           # classical_oracle (CPU pool)
if CAN_GPU:
    S4.run_flights_v4(CFG, todo, L, device=DEVICE, workers=GPU_WORKERS)     # physics and v2 arms, seed-major
else:
    print(f"{sum(not S4.flight_done(L, j) for j in todo if j['model']['kind'] != 'classical')} GPU flights wait "
          "for a GPU runtime")

[classical] 256/256 done, 0 to fly on 12 worker(s)
[flights] 9000/9000 done in 55 model runs (0 runs wait for training); 3 worker(s) on cuda


## 5. E3 training: R0 and 13 variants × 5 seeds

R0 is v2's recipe with one member; A1–A8 remove one v2 change from it; B1–B3 add one v2 change to v1 (v1's
recipe and v3's fair budget). Seed-major. Each run writes `02_models/<variant>/seed_<k>/`: `variant.json`
(the exact configuration and parameter count), the train state every 5 min (model, optimizer, scaler, step,
RNG state, loss history; a rerun resumes there), then `ckpt.pt` (marks it done; the train state is deleted),
`train_log.csv`, `latent_health.csv`, `F_curves.png`. Loads the 6,000 flights only if something is left to
train. About 17–20 h on an L4 for all 70 (v2-family ≈ 15–20 min each, B ≈ 5–10 min).

In [7]:
if RUN["E3_train"]:
    todo = [(v, s) for s in CFG.seeds for v in CFG.variants if not L.ckpt(v, s).exists()]   # seed-major
    print(f"{len(todo)} of {len(CFG.seeds) * len(CFG.variants)} trainings to do")
    if todo and CAN_GPU:
        need_data()
        for v, s in todo:
            t0 = time.time()
            r = AB.train_variant_seed(CFG, DATA, L, v, s, DEVICE, grad_ckpt=GRAD_CKPT)
            print(f"== {v} seed {s}: {'cached' if r['cached'] else f'trained in {(time.time() - t0) / 60:.1f} min'}")
    elif todo:
        print("training waits for a GPU runtime")
    p = L.model_dir("R0", CFG.seeds[0]) / "F_curves.png"
    if p.exists():
        display(Image(str(p)))

54 of 70 trainings to do
6000 flights: train/val/test = 4800/600/600 episodes (20s)
  [A1b_no_measured_aero seed 1 wm] resumed at step 8532/24000 from /content/drive/MyDrive/ReEntryAI_jepa_v4/02_models/A1b_no_measured_aero/seed_1/train_state_wm.pt
  [A1b_no_measured_aero seed 1 wm] step   9000/24000 T=48  pred=0.09311 sigreg=6.948 val_pred=0.09099 (39 ms/step, 5.3 min, ~10 min left)
  [A1b_no_measured_aero seed 1 wm] step   9500/24000 T=48  pred=0.08849 sigreg=6.758 val_pred=0.08249 (36 ms/step, 5.6 min, ~9 min left)
  [A1b_no_measured_aero seed 1 wm] step  10000/24000 T=48  pred=0.08199 sigreg=6.69 val_pred=0.0813 (36 ms/step, 5.9 min, ~8 min left)
  [A1b_no_measured_aero seed 1 wm] step  10500/24000 T=48  pred=0.07852 sigreg=6.754 val_pred=0.07611 (36 ms/step, 6.2 min, ~8 min left)
  [A1b_no_measured_aero seed 1 wm] step  11000/24000 T=48  pred=0.07329 sigreg=6.659 val_pred=0.07757 (36 ms/step, 6.5 min, ~8 min left)
  [A1b_no_measured_aero seed 1 wm] step  11500/24000 T=48  pred=0.06

KeyboardInterrupt: 

## 6. E3 flights: the 14 models × 5 seeds at 4.5 g

17,920 flights through the same lockstep runner, CRN and settings as v3. A model whose checkpoint does not
exist yet is skipped with a note (train it in section 5 first). About 15–25 h on an L4.

In [ ]:
if RUN["E3_fly"]:
    todo = [j for j in JOBS if j["stage"] == "E3"]
    if CAN_GPU:
        S4.run_flights_v4(CFG, todo, L, device=DEVICE, workers=GPU_WORKERS)
    else:
        print(f"{sum(not S4.flight_done(L, j) for j in todo)} E3 flights wait for a GPU runtime")

## 7. Open loop

Per model (R0, the 13 variants, and v3's v1) and seed: predicted vs true drogue point on the 600 held-out
test and 600 validation flights, from 400 to 50 s to go, with the commands actually flown
(`03_open_loop/footprint_<model>_seed<k>.csv`, along-track bias by density bin). Cached per file. Needs a
GPU (≈ 1 min per model-seed on an L4).

In [ ]:
if RUN["open_loop"]:
    models = list(CFG.variants) + ["v1"]
    ckpt = lambda m, s: L3.ckpt("v1", s) if m == "v1" else L.ckpt(m, s)
    todo = [(m, s) for s in CFG.seeds for m in models
            if ckpt(m, s).exists() and not (L.open_loop / f"footprint_{m}_seed{s}.csv").exists()]
    print(f"{len(todo)} model-seeds to evaluate open loop")
    if todo and CAN_GPU:
        eps = {"val": need_data()["val"], "test": DATA["test"]}
        for m, s in todo:
            t0 = time.time()
            A4.open_loop_seed(CFG, L, L3, eps, m, s, device=DEVICE)
            print(f"  {m} seed {s}: {time.time() - t0:.0f}s")
    elif todo:
        print("open loop waits for a GPU runtime")
    ol = A4.open_loop_table(L)
    if len(ol):
        display(ol[(ol.split == "test")].pivot_table(index="model", columns="ttg_s", values="err_median_km").round(2))

## 8. Analysis

v3 and v4 flights together: every table `05_analysis/tables/T01–T13`, figure `figures/F10–F18`,
`captions.json`, and the pre-registered criteria (§8: P1–P6, L1–L3, each A and B variant, R0-check, M),
with the hierarchical bootstrap (10,000 resamples) and Holm within each family. Bootstrap results are
cached in `05_analysis/cache/` by comparison and a hash of their input flights, so a rerun after new
flights recomputes only what changed. Runs on any runtime at any point; with a partly finished study it
reports what exists. ≈ 10–20 min the first time.

In [ ]:
if RUN["analysis"]:
    t0 = time.time()
    R = A4.analyze(CFG, L, L3)
    pd.set_option("display.max_colwidth", 140)
    display(R["T05"][["id", "question", "measured", "p_holm", "met"]])
    for k in ("F10", "F11", "F12", "F13", "F17"):
        if k in R["figures"]:
            display(Image(str(R["figures"][k])))
    print(f"analysis in {time.time() - t0:.0f}s")

## 9. Report

`06_report/SpaceJEPA_v4_report.pdf` (summary, RQ1–RQ4, methods, deviations from `v4_plan.md` §15, every
table), the status file, and a refreshed `00_study/manifest.csv` and `README.md`. Rebuilt every run (cheap).

In [ ]:
S4.status(CFG, L, S4.build_jobs(CFG, L, L3))
if RUN["report"]:
    print("report:", R4.build_report(L))
S4.update_manifest(L)
print(open(L.study / "README.md").read()[:3000])

## Done: release the runtime

Everything this notebook made is already on Drive. This cell flushes Drive and releases the Colab
runtime, so it stops using compute units (a failed cell does the same). To keep the runtime, set
`AUTO_DISCONNECT = False` in the setup cell before running.

In [ ]:
release_runtime("all cells finished")